# P5-2. 실험 기록표와 개선 루프

**PART 5 · DNN 미니프로젝트 · 학생용 (Student)**

---

## 학습 목표

1. 실험을 함수 하나로 반복 실행한다
2. 결과를 가설과 함께 표에 누적한다
3. 발표용 그림을 일괄 생성하고 최종 모델을 재현 검증한다

## 이 노트북 사용법 — 학생용 실습본

이 파일은 **핵심 코드만** 빈칸(`____`)으로 비워 둔 실습본입니다.

1. 코드 안의 `①`, `②` 번호와 힌트를 먼저 읽습니다.
2. `____`를 알맞은 값이나 코드로 바꿉니다.
3. `Shift + Enter`로 셀을 실행합니다.

| 오류 메시지 | 원인 |
|---|---|
| `NameError: name '____' is not defined` | 아직 채우지 않은 빈칸이 남아 있습니다. |
| `SyntaxError` | 빈칸과 함께 괄호나 따옴표를 지웠습니다. |

> `____`가 아닌 부분은 실행을 돕는 보조 코드입니다. 처음에는 수정하지 않아도 됩니다.


> 이 노트북은 `P5-1`을 실행한 뒤 이어서 실행한다.
> (같은 세션이 아니라면 P5-1의 1~6절 셀을 먼저 실행할 것)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import tensorflow as tf
layers = tf.keras.layers

plt.rcParams["axes.unicode_minus"] = False
tf.keras.utils.set_random_seed(42)
np.random.seed(42)
RANDOM_STATE = 42


> **입문자 안내**
> 내부 구현 전체를 외우지 않습니다. `____`가 있는 핵심 설정과 실행 순서에 집중하고, 긴 함수·클래스 코드는 실행용 보조 코드로 사용하세요.


In [ ]:
# P5-1의 결과 불러오기
import json

with open("baseline_reference.json", encoding="utf-8") as f:
    REF = json.load(f)

METRIC = REF["metric"]
TARGET = REF["target_to_beat"]
print(json.dumps(REF, ensure_ascii=False, indent=2))
print(f"\n넘어야 할 값: {METRIC} = {TARGET}")


> **주의** — 아래 셀은 P5-1에서 정의한 `X_train`, `evaluate`, `build_model`,
> `make_callbacks` 등을 그대로 사용한다.
> 커널을 새로 시작했다면 P5-1의 해당 셀들을 먼저 실행한다.

## 1. 실험 실행 함수 — 수작업을 없앤다

수작업으로는 20회를 넘기지 못한다. 함수로 만들면 100회도 가능하다.

In [ ]:
import time

EXPERIMENTS = []      # 모든 실험이 여기에 누적된다


def run_experiment(no, hypothesis, change, model_kwargs=None,
                   fit_kwargs=None, class_weight=None, threshold=0.5,
                   note_text=""):
    """실험 하나를 실행하고 기록에 추가한다.

    no          : 실험 번호
    hypothesis  : 왜 이것을 바꾸는가 (가장 중요한 항목)
    change      : 무엇을 바꿨는가
    """
    model_kwargs = model_kwargs or {}
    fit_kwargs = fit_kwargs or {}

    tf.keras.utils.set_random_seed(RANDOM_STATE)
    model = build_model(**model_kwargs)

    t0 = time.time()
    # 힌트: 학습 데이터와 검증 데이터, 콜백, class_weight를 넘긴다
    hist = model.fit(
        ____, # ①
        validation_data=____, # ②
        epochs=fit_kwargs.pop("epochs", 200),
        batch_size=fit_kwargs.pop("batch_size", 32),
        callbacks=make_callbacks(name=f"exp{no:02d}"),
        class_weight=class_weight,
        verbose=0, **fit_kwargs)
    sec = time.time() - t0

    # 힌트: 검증 데이터에 대한 예측 확률
    prob = ____  # ③ 
    # 힌트: P5-1에서 만든 evaluate 함수를 재사용한다
    scores = ____ # ④

    record = {
        "No": no,
        "가설": hypothesis,
        "변경 내용": change,
        **scores,
        "epochs": len(hist.history["loss"]),
        "params": model.count_params(),
        "sec": round(sec, 1),
        "판단": "",
        "비고": note_text,
    }
    EXPERIMENTS.append(record)

    delta = scores[METRIC] - TARGET
    verdict = "기준 초과" if delta > 0 else "기준 미달"
    print(f"[{no:02d}] {change}")
    print(f"     {METRIC} = {scores[METRIC]:.4f}  "
          f"(기준 대비 {delta:+.4f}, {verdict})  · {len(hist.history['loss'])} epoch")

    return model, hist, prob, scores


print("실험 함수 준비 완료")


## 2. 개선 실험 — 효과가 큰 것부터, 한 번에 하나씩

순서: 데이터 품질 → 입력 처리 → 모델 구조 → 정규화 → 학습 설정

In [ ]:
# 기준선 재기록 (표의 0번 행)
EXPERIMENTS.append({
    "No": 0, "가설": "(기준선)", "변경 내용": "Dense 64-32, 기본 설정",
    METRIC: REF["baseline_2_nn"], "epochs": "-", "params": "-", "sec": "-",
    "판단": "기준 확보", "비고": f"비딥러닝 최고: {REF['baseline_1_ml']}",
})

# 실험 1 — 모델 용량
m1, h1, p1, s1 = run_experiment(
    1,
    hypothesis="모델이 데이터에 비해 작아 패턴을 다 담지 못한다",
    change="은닉층 확대 (128-64-32)",
    model_kwargs=dict(hidden=(128, 64, 32)))


In [ ]:
# 실험 2 — 과적합 억제
m2, h2, p2, s2 = run_experiment(
    2,
    hypothesis="1번 모델이 과적합되었다 (train-val 격차가 크다)",
    change="1번 + Dropout 0.3",
    model_kwargs=dict(hidden=(128, 64, 32), dropout=0.3))

# 실험 3 — 학습 안정화
m3, h3, p3, s3 = run_experiment(
    3,
    hypothesis="층이 깊어져 학습이 불안정하다",
    change="2번 + BatchNormalization",
    model_kwargs=dict(hidden=(128, 64, 32), dropout=0.3, use_bn=True))


In [ ]:
# 실험 4 — 클래스 불균형 대응
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
cw = dict(enumerate(compute_class_weight("balanced", classes=classes, y=y_train)))
print("class_weight:", {k: round(v, 3) for k, v in cw.items()})

m4, h4, p4, s4 = run_experiment(
    4,
    hypothesis="소수 클래스를 놓치고 있어 재현율이 낮다",
    change="3번 + class_weight 적용",
    model_kwargs=dict(hidden=(128, 64, 32), dropout=0.3, use_bn=True),
    class_weight=cw)


In [ ]:
# 실험 5 — 임계값 조정 (재학습 없이 가능한 가장 값싼 개선)
from sklearn.metrics import f1_score

best_t, best_f1 = 0.5, 0.0
for t in np.arange(0.05, 0.96, 0.025):
    f1 = f1_score(y_val, (p4 > t).astype(int), zero_division=0)
    if f1 > best_f1:
        best_t, best_f1 = round(float(t), 3), f1

s5 = evaluate(y_val, p4, threshold=best_t)
EXPERIMENTS.append({
    "No": 5, "가설": "기본 임계값 0.5가 이 불균형에 맞지 않는다",
    "변경 내용": f"4번 모델 + 임계값 {best_t}",
    **s5, "epochs": "-", "params": m4.count_params(), "sec": 0.0,
    "판단": "", "비고": "재학습 없음",
})
print(f"[05] 임계값 {best_t} 적용 → {METRIC} = {s5[METRIC]:.4f} "
      f"(기준 대비 {s5[METRIC]-TARGET:+.4f})")


## 3. 실험 기록표 — 실패한 실험도 남긴다

In [ ]:
log = pd.DataFrame(EXPERIMENTS)

# 판단 열 자동 채우기 (팀이 직접 수정해도 된다)
for i, row in log.iterrows():
    if row["No"] == 0:
        continue
    prev = log.loc[log["No"] < row["No"], METRIC].max()
    delta = row[METRIC] - prev
    if delta > 0.01:
        log.loc[i, "판단"] = f"채택 ({delta:+.3f})"
    elif delta > 0:
        log.loc[i, "판단"] = f"보류 ({delta:+.3f}, 개선 미미)"
    else:
        log.loc[i, "판단"] = f"기각 ({delta:+.3f})"

cols = ["No", "가설", "변경 내용", METRIC, "recall", "precision",
        "epochs", "params", "판단", "비고"]
cols = [c for c in cols if c in log.columns]
log_view = log[cols]
log_view.to_csv("experiment_log.csv", index=False, encoding="utf-8-sig")
log_view


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
xs = log["No"].astype(str)
vals = log[METRIC].astype(float)
colors = ["#94A3B8"] + ["#2563EB"] * (len(vals) - 1)
bars = ax.bar(xs, vals, color=colors)
bars[int(vals.idxmax())].set_color("#059669")

ax.axhline(REF["baseline_1_ml"], ls="--", c="#D97706", lw=2,
           label=f"비딥러닝 기준 ({REF['baseline_1_ml']:.3f})")
ax.axhline(TARGET, ls=":", c="#DC2626", lw=2, label=f"넘어야 할 값 ({TARGET:.3f})")
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.008, f"{v:.3f}",
            ha="center", fontsize=9)
ax.set_xlabel("실험 번호"); ax.set_ylabel(METRIC)
ax.set_ylim(0, min(1.05, vals.max() * 1.25))
ax.legend(); ax.grid(alpha=0.3, axis="y")
ax.set_title("실험별 성능 추이 — 발표 핵심 그림")
plt.tight_layout(); plt.savefig("fig_experiment_progress.png", dpi=140,
                                bbox_inches="tight")
plt.show()


## 4. 최종 모델 확정 — 여기서 처음 test를 연다

In [ ]:
FINAL_MODEL = m4
FINAL_THRESHOLD = best_t

prob_test = FINAL_MODEL.predict(X_test, verbose=0).ravel()
final_score = evaluate(y_test, prob_test, threshold=FINAL_THRESHOLD)

print("=" * 55)
print("최종 테스트 성능 (test는 이번이 처음이자 마지막 사용)")
print("=" * 55)
for k, v in final_score.items():
    print(f"  {k:10s} {v}")

print("\n검증 성능과의 차이:")
val_final = evaluate(y_val, p4, threshold=FINAL_THRESHOLD)
for k in final_score:
    print(f"  {k:10s} val={val_final[k]:.4f}  test={final_score[k]:.4f}  "
          f"({final_score[k]-val_final[k]:+.4f})")


## 5. 발표용 그림 일괄 생성

In [ ]:
from sklearn.metrics import (confusion_matrix, roc_curve, auc,
                             precision_recall_curve, average_precision_score)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

# (1) 혼동행렬
pred_test = (prob_test > FINAL_THRESHOLD).astype(int)
cm = confusion_matrix(y_test, pred_test)
im = axes[0].imshow(cm, cmap="Blues")
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["예측 0", "예측 1"])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(["실제 0", "실제 1"])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, cm[i, j], ha="center", va="center", fontsize=16,
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
axes[0].set_title("혼동행렬 (test)")

# (2) ROC
fpr, tpr, _ = roc_curve(y_test, prob_test)
axes[1].plot(fpr, tpr, lw=2.5, c="#2563EB", label=f"AUC = {auc(fpr,tpr):.3f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR")
axes[1].set_title("ROC 커브"); axes[1].legend(); axes[1].grid(alpha=0.3)

# (3) PR
prec, rec, _ = precision_recall_curve(y_test, prob_test)
ap = average_precision_score(y_test, prob_test)
axes[2].plot(rec, prec, lw=2.5, c="#059669", label=f"AP = {ap:.3f}")
axes[2].axhline(y_test.mean(), ls="--", c="k", lw=1,
                label=f"무작위 기준 ({y_test.mean():.3f})")
axes[2].set_xlabel("Recall"); axes[2].set_ylabel("Precision")
axes[2].set_title("PR 커브"); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("fig_final_evaluation.png", dpi=140, bbox_inches="tight")
plt.show()


In [ ]:
# 학습곡선 — 베이스라인과 최종 모델 비교
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
axes[0].plot(h1.history["val_loss"], lw=2, c="#94A3B8", label="실험 1 (정규화 없음)")
axes[0].plot(h4.history["val_loss"], lw=2.5, c="#059669", label="실험 4 (최종)")
axes[0].set_title("val_loss"); axes[0].set_xlabel("epoch")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(h4.history["loss"], lw=2, c="#2563EB", label="train")
axes[1].plot(h4.history["val_loss"], lw=2, c="#DC2626", label="val")
axes[1].set_title("최종 모델 학습곡선"); axes[1].set_xlabel("epoch")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("fig_learning_curves.png", dpi=140, bbox_inches="tight")
plt.show()
print("저장된 그림: fig_experiment_progress.png, fig_final_evaluation.png, "
      "fig_learning_curves.png")


## 6. 오류 분석 — 무엇을 틀렸는가

In [ ]:
fn_idx = np.where((y_test == 1) & (pred_test == 0))[0]      # 놓친 양성
fp_idx = np.where((y_test == 0) & (pred_test == 1))[0]      # 오경보

print(f"FN (놓친 양성): {len(fn_idx)}건")
print(f"FP (오경보)  : {len(fp_idx)}건")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(prob_test[y_test == 0], bins=40, alpha=0.65,
             color="#2563EB", label="실제 음성")
axes[0].hist(prob_test[y_test == 1], bins=40, alpha=0.65,
             color="#DC2626", label="실제 양성")
axes[0].axvline(FINAL_THRESHOLD, c="k", ls="--", lw=2,
                label=f"임계값 {FINAL_THRESHOLD}")
axes[0].set_xlabel("예측 확률"); axes[0].set_ylabel("건수")
axes[0].set_title("예측 확률 분포"); axes[0].legend(); axes[0].grid(alpha=0.3)

if len(fn_idx) > 0:
    axes[1].hist(prob_test[fn_idx], bins=25, color="#D97706")
    axes[1].axvline(FINAL_THRESHOLD, c="k", ls="--", lw=2)
    axes[1].set_xlabel("예측 확률"); axes[1].set_title("놓친 양성(FN)의 확률 분포")
    axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig("fig_error_analysis.png", dpi=140,
                                bbox_inches="tight")
plt.show()

if len(fn_idx) > 0:
    near = (prob_test[fn_idx] > FINAL_THRESHOLD - 0.1).sum()
    print(f"\nFN 중 임계값 근처(-0.1 이내): {near}건 ({near/len(fn_idx):.1%})")
    print("→ 임계값을 조금만 낮추면 잡을 수 있는 사례가 이만큼이다")


## 7. 재현성 확인과 산출물 저장

In [ ]:
import json

FINAL_MODEL.save("final_model.keras")

reloaded = tf.keras.models.load_model("final_model.keras")
p_reload = reloaded.predict(X_test, verbose=0).ravel()
diff = np.abs(prob_test - p_reload).max()
print(f"저장 모델 재현 확인 — 최대 차이: {diff:.10f}")
assert diff < 1e-6, "재현 실패"

result = {
    "project": REF["project"],
    "metric": METRIC,
    "baseline_dummy": REF["baseline_0_dummy"],
    "baseline_ml": REF["baseline_1_ml"],
    "baseline_nn": REF["baseline_2_nn"],
    "final_val": val_final[METRIC],
    "final_test": final_score[METRIC],
    "improvement_vs_baseline": round(final_score[METRIC] - REF["baseline_2_nn"], 4),
    "threshold": FINAL_THRESHOLD,
    "n_experiments": len(EXPERIMENTS) - 1,
    "success_criterion": REF["success_criterion"],
    "criterion_met": bool(final_score[METRIC] > TARGET),
}
with open("final_result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("\n" + json.dumps(result, ensure_ascii=False, indent=2))


In [ ]:
# 제출 산출물 점검
import os

required = {
    "final_model.keras": "최종 모델",
    "experiment_log.csv": "실험 기록표",
    "baseline_scores.csv": "베이스라인 기록",
    "final_result.json": "최종 결과 요약",
    "fig_experiment_progress.png": "실험 추이 그림",
    "fig_final_evaluation.png": "최종 평가 그림",
    "fig_learning_curves.png": "학습곡선",
    "fig_error_analysis.png": "오류 분석",
}

print("=== 제출 산출물 점검 ===")
missing = []
for path, desc in required.items():
    exists = os.path.exists(path)
    mark = "O" if exists else "X"
    size = f"{os.path.getsize(path)/1024:.1f} KB" if exists else "-"
    print(f"  [{mark}] {desc:20s} {path:32s} {size}")
    if not exists:
        missing.append(path)

print("\n추가로 직접 작성할 것: README.md, 발표자료(10장 내외)")
if missing:
    print("누락:", missing)
else:
    print("\n모든 자동 생성 산출물이 준비되었다.")


### 확인 질문

- 가장 효과가 컸던 실험은 몇 번이고 그 가설은 무엇이었는가
- 기각된 실험에서 무엇을 배웠는가
- val과 test 성능 차이가 크다면 무엇을 의심해야 하는가
- 비딥러닝 베이스라인을 넘지 못했다면 발표에서 어떻게 말하겠는가

---

## 정리

- 실험은 함수로 자동화한다. 수작업으로는 20회를 넘기지 못한다.
- 기록표의 핵심은 "무엇을 바꿨는가"가 아니라 **"왜 그것을 바꿨는가"** 이다.
- 실패한 실험이 빠진 기록은 기록이 아니다.
- test는 최종 확정 후 한 번만 연다.
- 재현 검증까지 마쳐야 산출물이 완성된다.
